In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
! pip install -q kagglehub scikit-learn pillow

In [ ]:
import os
import gc
import random
import pathlib
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

In [ ]:
try:
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver()

    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)

    strategy = tf.distribute.TPUStrategy(resolver)

    print("✅ TPU detected")
    print("TPU workers:", resolver.cluster_spec().as_dict())
    print("Replicas:", strategy.num_replicas_in_sync)

except ValueError:
    print("⚠️ TPU not detected.")
    print("Kaggle → Settings → Accelerator → TPU select করো.")

    strategy = tf.distribute.get_strategy()

print("Strategy:", type(strategy).__name__)

In [ ]:
SEARCH_TERMS = [
    "bornomala bengali handwritten",
    "bornomala handwritten",
    "bornomala"
]

results = []

for term in SEARCH_TERMS:
    url = "https://www.kaggle.com/api/v1/datasets/list"

    response = requests.get(
        url,
        params={
            "search": term,
            "page": 1
        },
        timeout=30
    )

    if response.status_code == 200:
        data = response.json()

        for item in data:
            results.append({
                "title": item.get("title"),
                "ref": item.get("ref"),
                "size": item.get("totalBytes")
            })


results_df = pd.DataFrame(results)

if len(results_df):
    results_df = results_df.drop_duplicates("ref")

results_df

In [ ]:
DATASET_REF = None

if len(results_df):

    exact_candidates = results_df[
        results_df["title"]
        .str.lower()
        .str.contains("bornomala", na=False)
    ]

    print(exact_candidates)

    if len(exact_candidates) > 0:
        DATASET_REF = exact_candidates.iloc[0]["ref"]


print("\nSelected dataset:")
print(DATASET_REF)

In [ ]:
if DATASET_REF is None:
    raise RuntimeError(
        "Public Bornomala Kaggle dataset not found \n"
        "Exact Kaggle dataset handle replace "
    )

DATA_PATH = kagglehub.dataset_download(DATASET_REF)

print("✅ Dataset downloaded")
print("Path:", DATA_PATH)

In [ ]:
for root ,dirs, files in os.walk(DATA_PATH): 
    level = root.replace(DATA_PATH,"").count(os.sep)
    if level > 2 : 
        continue 
    indent = "" + level 

    print(f"{indent}{os.path.basename(root)}")


    for f in files[:5]: 
        print(f"{indent} {f}")
    